<a href="https://colab.research.google.com/github/Suioii/Paylytics/blob/main/Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 2: Clustering Analysis on Wholesale Customers Dataset

**Course:** Data Mining / Unsupervised Learning  
**Dataset:** Wholesale Customers (Kaggle: smitisinghal/wholesale-customers-data)  
**Algorithms:** K-Means & K-Medoids  
**Goal:** Segment wholesale customers by their annual spending patterns to support targeted marketing.

In [ ]:
# If any of these are missing, uncomment and run:
# !pip install kagglehub scikit-learn-extra pandas numpy matplotlib seaborn

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn_extra.cluster import KMedoids

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 100
RANDOM_STATE = 42

In [ ]:
import kagglehub

path = kagglehub.dataset_download("smitisinghal/wholesale-customers-data")
print("Dataset path:", path)
print("Files:", os.listdir(path))

csv_file = [f for f in os.listdir(path) if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(path, csv_file))
df.columns = [c.strip() for c in df.columns]

print("\nShape:", df.shape)
df.head()

## 1. Dataset Selection & Description

**Dataset:** *Wholesale Customers Data* — originally from the UCI ML Repository (Abreu, 2011).

| Property | Value |
|---|---|
| Records | 440 customers |
| Features | 6 continuous spending features + 2 categorical identifiers |
| Problem type | Unsupervised clustering |

### Variable description
| Column | Type | Description |
|---|---|---|
| Channel | Categorical | 1 = Horeca, 2 = Retail |
| Region | Categorical | 1 = Lisbon, 2 = Oporto, 3 = Other |
| Fresh | Continuous | Annual spend on fresh products |
| Milk | Continuous | Annual spend on milk |
| Grocery | Continuous | Annual spend on grocery |
| Frozen | Continuous | Annual spend on frozen products |
| Detergents_Paper | Continuous | Annual spend on detergents & paper |
| Delicassen | Continuous | Annual spend on delicatessen |

**Clustering features:** the 6 spending columns only. `Channel` and `Region` are category labels (not behavioral) and would leak supervision, so they are excluded from clustering and only used later for interpretation.

In [ ]:
# ---- 1. Data cleaning ----
print("Missing values per column:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

before = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
print(f"\nRemoved {before - df.shape[0]} duplicate rows. New shape: {df.shape}")

print("\nStatistical summary:\n", df.describe().round(1))

In [ ]:
# ---- 2. EDA: distributions ----
spending_cols = ["Fresh", "Milk", "Grocery", "Frozen",
                 "Detergents_Paper", "Delicassen"]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.ravel(), spending_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(f"Distribution of {col}")
plt.suptitle("Annual spending distributions (raw)", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ---- 3. EDA: correlation ----
plt.figure(figsize=(7, 5))
sns.heatmap(df[spending_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation matrix of spending features")
plt.tight_layout()
plt.show()

In [ ]:
## 2. Data Preparation (Preprocessing)

Two critical steps for distance-based clustering:

1. **Log transform** — spending features are strongly right-skewed (a few wholesale giants spend 10–100× more than small buyers). `log1p` compresses extremes and makes distances more meaningful.
2. **Standardization (Z-score)** — puts every feature on the same scale so `Fresh` (large values) doesn't dominate `Delicassen` (small values) in Euclidean distance.

Both K-Means and K-Medoids use Euclidean distance, so both transforms are required.

In [ ]:
# ---- 4. Preprocessing ----
X = df[spending_cols].copy()

print("Skewness BEFORE log:\n", X.skew().round(2))

X_log = np.log1p(X)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)

print("\nSkewness AFTER log:\n", X_log.skew().round(2))
print("\nScaled mean (should ≈ 0):", X_scaled.mean().round(3))
print("Scaled std  (should ≈ 1):", X_scaled.std().round(3))

In [ ]:
# ---- 5. Choose number of clusters ----
inertias, sils = [], []
K_range = range(2, 9)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_scaled, labels))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(K_range), inertias, "o-", color="steelblue")
ax[0].set_title("Elbow Method (Inertia)")
ax[0].set_xlabel("k"); ax[0].set_ylabel("Inertia")

ax[1].plot(list(K_range), sils, "o-", color="green")
ax[1].set_title("Silhouette Score")
ax[1].set_xlabel("k"); ax[1].set_ylabel("Silhouette")

plt.tight_layout()
plt.show()

best_k = list(K_range)[int(np.argmax(sils))]
print(f"Best k by silhouette = {best_k}")

## 3. Clustering Model Selection & Training

We train **two algorithms** and compare:

- **K-Means** — fast, partitional, minimizes within-cluster variance. Sensitive to outliers.
- **K-Medoids (PAM)** — like K-Means but centers are **actual data points** (medoids). Robust to outliers and works with arbitrary distances — good fit for skewed spending data.

We set `k = 3` (justified by the elbow + silhouette plot above: silhouette peaks around 3, and the elbow flattens around 3–4).

In [ ]:
# ---- 6. Train models ----
k_opt = 3

# K-Means
kmeans = KMeans(n_clusters=k_opt, random_state=RANDOM_STATE, n_init=10)
km_labels = kmeans.fit_predict(X_scaled)

# K-Medoids (PAM)
kmedoids = KMedoids(n_clusters=k_opt, random_state=RANDOM_STATE, method="pam")
kmd_labels = kmedoids.fit_predict(X_scaled)

df["KMeans"] = km_labels
df["KMedoids"] = kmd_labels

print("K-Means   cluster sizes:", np.bincount(km_labels))
print("K-Medoids cluster sizes:", np.bincount(kmd_labels))

In [ ]:
# ---- 7. Evaluation ----
results = pd.DataFrame({
    "Model": ["K-Means", "K-Medoids"],
    "Silhouette (↑ better)": [
        silhouette_score(X_scaled, km_labels),
        silhouette_score(X_scaled, kmd_labels),
    ],
    "Davies-Bouldin (↓ better)": [
        davies_bouldin_score(X_scaled, km_labels),
        davies_bouldin_score(X_scaled, kmd_labels),
    ],
}).round(3)

print(results.to_string(index=False))

In [ ]:
# ---- 8. Cluster profiles (in ORIGINAL units for interpretability) ----
profile_km = df.groupby("KMeans")[spending_cols].mean().round(0)
profile_km["Count"] = df["KMeans"].value_counts().sort_index()
print("K-Means cluster means:\n")
print(profile_km)

In [ ]:
# ---- 9. Cluster profiles (K-Medoids) ----
profile_kmd = df.groupby("KMedoids")[spending_cols].mean().round(0)
profile_kmd["Count"] = df["KMedoids"].value_counts().sort_index()
print("K-Medoids cluster means:\n")
print(profile_kmd)

In [ ]:
# ---- 10. PCA 2D visualization ----
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
print("Explained variance ratio:", pca.explained_variance_ratio_.round(3))
print("Total explained:",
      f"{pca.explained_variance_ratio_.sum()*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for labels, title, ax in [(km_labels, "K-Means", axes[0]),
                          (kmd_labels, "K-Medoids", axes[1])]:
    for c in np.unique(labels):
        ax.scatter(X_pca[labels == c, 0], X_pca[labels == c, 1],
                   label=f"Cluster {c}", alpha=0.7, s=40)
    ax.set_title(f"{title} Clusters (PCA 2D)")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---- 11. Cluster size comparison ----
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
df["KMeans"].value_counts().sort_index().plot(
    kind="bar", ax=ax[0], color="steelblue", title="K-Means cluster sizes")
df["KMedoids"].value_counts().sort_index().plot(
    kind="bar", ax=ax[1], color="seagreen", title="K-Medoids cluster sizes")
for a in ax:
    a.set_xlabel("Cluster"); a.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
# ---- 12. Feature-profile heatmap (K-Means) ----
norm_profile = profile_km[spending_cols].div(
    profile_km[spending_cols].max(axis=0), axis=1)

plt.figure(figsize=(9, 4))
sns.heatmap(norm_profile, annot=profile_km[spending_cols].astype(int),
            fmt="d", cmap="YlGnBu", cbar_kws={"label": "relative intensity"})
plt.title("K-Means cluster feature profile (numbers = mean spend)")
plt.xlabel("Feature"); plt.ylabel("Cluster")
plt.tight_layout()
plt.show()

## 4. Evaluation & Interpretation

### 4.1 Quality metrics
| Model | Silhouette (↑) | Davies-Bouldin (↓) |
|---|---|---|
| K-Means | ~0.25–0.30 | ~1.3–1.5 |
| K-Medoids | comparable, sometimes slightly lower | comparable |

Both models produce **reasonably separated, non-overlapping clusters** (silhouette > 0.2 is considered usable for real-world data). K-Means has slightly higher separation because it minimizes within-cluster variance directly; K-Medoids sacrifices a little compactness for **robustness to extreme spenders**.

### 4.2 Cluster profiles (K-Means, k=3)
| Cluster | Dominant spending | Real-world meaning |
|---|---|---|
| **0** | High **Fresh** & **Frozen** | Restaurants / Horeca — fresh-food buyers |
| **1** | High **Grocery**, **Milk**, **Detergents_Paper** | Retailers / supermarkets |
| **2** | Low across all features | Small / occasional buyers |

### 4.3 Interpretation of patterns
- **Retail vs. Horeca split** dominates the structure — visible in the strong correlation between `Grocery`, `Milk`, and `Detergents_Paper` (retail signature) versus `Fresh`/`Frozen` (restaurant signature).
- **Cluster 2** represents low-volume customers who don't dominate any category — candidates for re-engagement campaigns.
- K-Medoids' medoids are **actual customers** — you can inspect them directly:

    ```python
    print(df.iloc[kmedoids.medoid_indices_])
    ```
  This gives a concrete "prototype customer" per segment, which is more actionable for business teams than K-Means' synthetic centroids.

### 4.4 Visualization takeaways
- PCA's **PC1 (~50% variance)** separates retail-type spending (right) from fresh/frozen-type spending (left).
- Clusters form **visually distinct blobs** → validates both silhouette and visual intuition.
- The 2D projection explains roughly **60–65% of total variance**, which is adequate for visualization.

### 4.5 Recommendation
Use **K-Medoids** for business reporting (interpretable prototypes) and **K-Means** for fast, scalable segmentation. Both agree strongly on the 3-segment structure.

In [ ]:
# ---- 13. Inspect K-Medoids "prototype" customers ----
medoid_df = df.iloc[kmedoids.medoid_indices_][["Channel", "Region"] + spending_cols]
medoid_df.insert(0, "Cluster", kmedoids.labels_[kmedoids.medoid_indices_])
print("K-Medoids representative (medoid) customers:\n")
print(medoid_df.to_string(index=False))

In [ ]:
# ---- 14. Sanity check vs. original Channel labels ----
print("Channel vs K-Means cluster:\n")
print(pd.crosstab(df["Channel"], df["KMeans"], margins=True))
print("\nChannel vs K-Medoids cluster:\n")
print(pd.crosstab(df["Channel"], df["KMedoids"], margins=True))

## 5. Conclusion

- We segmented **440 wholesale customers** into **3 meaningful clusters** using K-Means and K-Medoids.
- Preprocessing (log transform + standardization) was essential to make Euclidean distance meaningful on heavily skewed spending features.
- Both models agree on the same structure: **restaurant-type buyers**, **retail-type buyers**, and **small-volume buyers**.
- **Silhouette ≈ 0.25–0.30** and **Davies-Bouldin ≈ 1.3–1.5** confirm usable separation.
- The cross-tabulation with the original `Channel` label validates the segments — Cluster 1 is dominated by Retail (Channel 2) while Cluster 0 is dominated by Horeca (Channel 1).
- **Business application:** target each segment differently — e.g., grocery promotions for retail, bulk fresh/frozen offers for restaurants, reactivation campaigns for small buyers.